In [2]:
import sys

import rtde_control
import rtde_receive
from magpie_control import ur5
import importlib
importlib.reload(ur5)
import time
import numpy as np
from magpie_control.gripper import Gripper
from magpie_control import gripper
importlib.reload(gripper)
from magpie_control import poses

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [3]:
ROBOT_IP = "192.168.0.4"
robot = ur5.UR5_Interface(ROBOT_IP, 
                            # freq=6, # 6Hz frequency
                            record=False,
                            provide_gripper=True,
                            provide_ft_sensor=True,)
robot.start()
robot.start_ft_sensor()
G = robot.gripper
G.reset_parameters()
# np.save("scalingforce/home.npy", robot.home)
# pose = np.load("scalingforce/pose.npy")

Found Dynamixel Port:
/dev/ttyACM1

Succeeded to open the port
Succeeded to change the baudrate
[RxPacketError] Overload error!
[RxPacketError] Overload error!


In [6]:
robot.toggle_teach_mode()

In [13]:
robot.stop()

Successfully closed port


In [4]:
pose = robot.get_tcp_pose()
pose

array([[-0.86 ,  0.473, -0.195, -0.171],
       [ 0.495,  0.865, -0.085, -0.351],
       [ 0.129, -0.169, -0.977,  0.419],
       [ 0.   ,  0.   ,  0.   ,  1.   ]])

In [12]:
robot.moveL(pose, asynch=False)
# robot.moveL_translation_tooltip(pose, asynch=False)


Tooltip: [0.012 0.006 0.211]
Goal Pos: [-0.741 -0.661  0.335]


In [39]:
robot.moveL(robot.home, asynch=False)

RTDEControlInterface: RTDE control script is not running!


In [3]:
motion_plan = np.load("scalingforce/motion_plan.npy", allow_pickle=True).item()
G.reset_and_close_gripper()
motion_plan

{'position_direction': [-1, 0, 0],
 'position_goal': [0.1, 0.0, 0.0],
 'force': [3.0, 0.0, 0.5],
 'grasp_force': 7.5,
 'duration': 5.0}

In [12]:
robot.ctrl.speedL([0.00, -0.00, 0.00, 0.0, 0.0, 0.0], 0.5, 0.25)


True

In [7]:
goal_delta = np.array(motion_plan['position_goal'])
direction = np.array(motion_plan['position_direction'])
sign = np.where(direction < 0, -1, 1)
force = np.array(motion_plan['force']) * sign
wrench_goal = np.hstack((force, np.zeros(3)))
init_cmd = np.hstack((force / -300, np.zeros(3)))
duration = motion_plan['duration']
# duration = 2
grasp_force = motion_plan['grasp_force']
P = 0.0003 # stiffness gain, default is 0.0005
import asyncio

# await asyncio.gather(
#     # G.deligrasp_async(40, grasp_force, 2, 0.4, complete=True, debug=False),
#     robot.force_position_control_async(wrench=wrench_goal, goal_delta=goal_delta, 
#                    init_cmd=init_cmd, tolerance=0.01, duration=duration, p=P, control_type="proportional")
#                 #    init_cmd=init_cmd, tolerance=0.01, duration=duration, control_type="bang_bang")
# )
robot.force_position_control(wrench=wrench_goal, goal_delta=goal_delta, grasp_force=grasp_force,
                init_cmd=init_cmd, tolerance=0.01, duration=duration, p=P, control_type="proportional")

robot.ctrl.speedL([0.00, -0.00, 0.00, 0.0, 0.0, 0.0], 0.5, 0.25)


[RxPacketError] Overload error!
[RxPacketError] Overload error!
[RxPacketError] Overload error!
[RxPacketError] Overload error!
[RxPacketError] Overload error!
[RxPacketError] Overload error!
Error reading from OptoForce!


True

In [45]:
G.reset_packet_overload()
G.open_gripper()

[RxPacketError] Overload error!
[RxPacketError] Overload error!
[RxPacketError] Overload error!


In [32]:
# dict of z-value vs z-offset
# 0.278: 0.1

In [5]:
tmat = np.array([[ 0.50676157,  0.63485957,  0.58322041,  0.03397274],
        [ 0.46190666, -0.77117344,  0.43810246, -0.04689727],
        [-0.72789762, -0.0473799 ,  0.68404692,  0.37931564],
        [ 0.        ,  0.        ,  0.        ,  1.        ]])

z_offset = 0.10
# z_offset += -0.045

# eye = np.eye(4)
# eye[2, 3] = -z_offset

# tmat = tmat @ eye
print(tmat)
# aperture = 37
# # z_offset = G.aperture_to_z(aperture)/1000.0

# mmc = copy.deepcopy(tmat[:3, 3])
# gripper_offset = [0.012, 0.006, 0.221] # x, y, z offset
# grasp_pose = mmc - gripper_offset

# # make identity matrix with grasp_pose as translation
# tmat_offset = np.eye(4)
# tmat_offset[:3, 3] = grasp_pose
# print(tmat_offset)

# tmat[:3, 3] = [0, 0, 0]
homePose = None

[[ 0.507  0.635  0.583  0.034]
 [ 0.462 -0.771  0.438 -0.047]
 [-0.728 -0.047  0.684  0.379]
 [ 0.     0.     0.     1.   ]]


In [3]:
tmat = np.array([[ 0.8041608 , -0.3825053 ,  0.45498912,  0.0416059 ],
        [-0.36299512, -0.92215233, -0.13367728, -0.02677324],
        [-0.47070155,  0.0576608 ,  0.88040632,  0.47194844],
        [ 0.        ,  0.        ,  0.        ,  1.        ]])

In [5]:
# move gripper to cartesian pos
# OR orient gripper in place
sleepRate = 1.5
# time.sleep(sleepRate)
ur = ur5.UR5_Interface()
try:
    ur.start()
    # ur.open_gripper()
    homePose = ur.get_tcp_pose()
    # wristPose = ur.getPose()
    # wrist = np.array(wristPose)
    # # tmat[:3, 3] -= gripper_offset
    # print(wrist)
    # print(wrist @ tmat_offset)
    # desired_pose = wrist @ tmat_offset
    # ur.moveL(desired_pose)
    ur.move_tcp_cartesian(tmat)
    time.sleep(sleepRate * 1.2)# offset_pose = gt.get_world_frame(mmc, ur, -offset)
    # desiredPose = np.array(wristPose)
    # desiredPose[:3, 3] += offset_pose
    # print(desiredPose)
    ur.stop()
except Exception as e:
    ur.stop()
    raise(e)

Succeeded to open the port
Succeeded to change the baudrate


In [12]:
ur = ur5.UR5_Interface()
try:
    ur.start()
    ur.close_gripper()
    # ur.open_gripper()
    time.sleep(sleepRate)
    ur.stop()
except Exception as e:
    ur.stop()
    raise(e)

Succeeded to open the port
Succeeded to change the baudrate


In [6]:
# move home
ur = ur5.UR5_Interface()
try:
    ur.start()
    ur.open_gripper()
    time.sleep(sleepRate)
    # homePose = np.array([[-0.024, -0.998, -0.062, -0.261],
    #                     [-0.999,  0.021,  0.035, -0.162],
    #                     [-0.033,  0.063, -0.997,  0.221],
    #                     [ 0.   ,  0.   ,  0.   ,  1.   ]])
    ur.moveL(homePose)
    # gt_home = ur.getPose()
    # print(np.array(gt_home))
    time.sleep(sleepRate * 1.1)
    ur.stop()
except Exception as e:
    ur.stop()

Succeeded to open the port
Succeeded to change the baudrate
